# 🚇 MTA Real-Time Train Timing Board

This notebook fetches live departure data from the MTA GTFS-Realtime API and displays it as ASCII art.

**Before running:** make sure you've set your MTA API key either:
- In `config.py` → `MTA_API_KEY = "your_key_here"`, or
- As an environment variable `MTA_API_KEY` (safer — keeps it out of the notebook)

Get a free key at https://api.mta.info/#/signup

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
# Load dependencies and optionally override the API key from an env variable.
# Run this cell once at startup.

import os
import time
import datetime
import pytz
from IPython.display import clear_output, HTML, display

import config           # all settings live here
import mta_fetcher      # feed fetching + parsing
import ascii_renderer   # display logic

# ── Optional: load API key from environment variable ──────────────────────────
# This keeps your key out of the notebook file itself (safer for git/sharing).
# Set it in your shell with: export MTA_API_KEY="your_key"
env_key = os.environ.get("MTA_API_KEY")
if env_key:
    config.MTA_API_KEY = env_key
    print("✓ API key loaded from environment variable")
elif config.MTA_API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️  No API key set! Edit config.py or set the MTA_API_KEY env variable.")
else:
    print("✓ API key loaded from config.py")

print(f"  Available lines: {list(config.LINE_CONFIGS.keys())}")

In [ ]:
# ── Cell 2: Choose a Line ─────────────────────────────────────────────────────
# Set ACTIVE_LINE to any line defined in config.LINE_CONFIGS.
# Currently only "M" is fully configured. See config.py to add more.

ACTIVE_LINE = "M"    # ← change this to switch lines

line_config = config.LINE_CONFIGS[ACTIVE_LINE]
feed_key    = line_config["feed"]
print(f"Selected: [{ACTIVE_LINE}] train  |  Feed: {feed_key}")

In [ ]:
# ── Cell 3: Single Fetch (terminal output) ────────────────────────────────────
# Run this cell for a one-shot snapshot in the terminal below.
# Great for testing your API key before running the live loop.

print("Fetching feed...")
feed       = mta_fetcher.fetch_feed(feed_key)
departures = mta_fetcher.get_departures(feed, line_config)
ascii_renderer.render_terminal(departures, line_config)

In [ ]:
# ── Cell 4: Live Refresh Loop (terminal) ──────────────────────────────────────
# Refreshes every REFRESH_INTERVAL_SECONDS seconds.
# Stop it by interrupting the kernel (■ button or Kernel → Interrupt).

print(f"Starting live display. Refreshes every {config.REFRESH_INTERVAL_SECONDS}s.")
print("Interrupt the kernel to stop.\n")

try:
    while True:
        clear_output(wait=True)   # wipe previous output before redrawing
        try:
            feed       = mta_fetcher.fetch_feed(feed_key)
            departures = mta_fetcher.get_departures(feed, line_config)
            ascii_renderer.render_terminal(departures, line_config)
        except Exception as e:
            # Don't crash the loop on a transient network error
            print(f"⚠️  Error fetching feed: {e}")
            print(f"   Retrying in {config.REFRESH_INTERVAL_SECONDS}s...")

        time.sleep(config.REFRESH_INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\nStopped.")

In [ ]:
# ── Cell 5: Live Refresh Loop (rich HTML in notebook) ─────────────────────────
# Renders the HTML version inside the notebook output cell.
# Useful to preview how the Neocities page will look.

print(f"Starting HTML preview. Refreshes every {config.REFRESH_INTERVAL_SECONDS}s.")
print("Interrupt the kernel to stop.\n")

try:
    while True:
        clear_output(wait=True)
        try:
            feed       = mta_fetcher.fetch_feed(feed_key)
            departures = mta_fetcher.get_departures(feed, line_config)
            html_out   = ascii_renderer.render_html(departures, line_config)
            display(HTML(html_out))
        except Exception as e:
            display(HTML(f'<p style="color:red">⚠️ Error: {e}</p>'))

        time.sleep(config.REFRESH_INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\nStopped.")

In [ ]:
# ── Cell 6: Save HTML to File ─────────────────────────────────────────────────
# Saves the current snapshot as index.html in this directory.
# Upload this file to Neocities to host it publicly.
#
# Tip: the <meta http-equiv="refresh" content="30"> tag in the HTML means
# the page auto-reloads every 30 seconds in the browser, but it will always
# show the data that was current when you last ran this cell and uploaded.
# For truly live data on Neocities, re-run this cell + re-upload regularly,
# or run the export_loop cell below for scheduled exports.

OUTPUT_FILE = "index.html"

feed       = mta_fetcher.fetch_feed(feed_key)
departures = mta_fetcher.get_departures(feed, line_config)
html_out   = ascii_renderer.render_html(departures, line_config)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html_out)

print(f"✓ Saved to {OUTPUT_FILE}")
print(f"  Upload this file to Neocities to publish your timing board.")

In [ ]:
# ── Cell 7: Scheduled HTML Export Loop ────────────────────────────────────────
# Continuously re-fetches and re-saves index.html on a schedule.
# Pair this with a Neocities CLI upload script to keep the page live.
# See README.md → "Keeping Neocities Updated" for the upload approach.

EXPORT_INTERVAL = 60   # seconds between each export (higher = fewer API calls)

print(f"Export loop started. Saving index.html every {EXPORT_INTERVAL}s.")
print("Interrupt the kernel to stop.\n")

try:
    while True:
        try:
            feed       = mta_fetcher.fetch_feed(feed_key)
            departures = mta_fetcher.get_departures(feed, line_config)
            html_out   = ascii_renderer.render_html(departures, line_config)

            with open("index.html", "w", encoding="utf-8") as f:
                f.write(html_out)

            ts = datetime.datetime.now().strftime("%H:%M:%S")
            print(f"[{ts}] ✓ index.html updated")

        except Exception as e:
            print(f"⚠️  Error: {e}")

        time.sleep(EXPORT_INTERVAL)

except KeyboardInterrupt:
    print("\nExport loop stopped.")